In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

In [ ]:
#path
table_multi_path = r"C:\Users\anacs\OneDrive\Área de Trabalho\ic-iag\Work 2\ALL_MorphoSPLUS_GalfitM_output_splus.csv"
labels_table_path = r"C:/Users/anacs/OneDrive\Área de Trabalho\ic-iag\Work\tabela_filtrada.csv"

#read
df_tab = pd.read_csv(table_multi_path)
df_labels = pd.read_csv(labels_table_path)

df_labels["ID"] = df_labels["ID"].astype(str)
df_labels["label"] = df_labels["type"].apply(lambda x: 0 if int(x) == 0 else 1)

#merge
df_tab["ID_1"] = df_tab["ID_1"].astype(str)

df = df_tab.merge(
    df_labels[["ID", "label"]],
    left_on="ID_1",
    right_on="ID",
    how="inner"
).drop(columns=["ID"])

df["label"] = df["label"].astype(int)

print("Final table:", df.shape)
print(df["label"].value_counts())

#convert to numeric (except text)
colunas_texto = ["source_folder", "source_file", "ID_1", "Field_ID", "Dir_", "Field"]

for col in df.columns:
    if col in colunas_texto:
        continue

    df[col] = df[col].astype(str)
    df[col] = df[col].str.replace("*", "", regex=False)
    df[col] = df[col].str.replace(",", ".", regex=False)
    df[col] = df[col].str.strip()
    df[col] = pd.to_numeric(df[col], errors="coerce")

#numeric columns
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols.remove("label")

print("Numeric cols:", len(numeric_cols))

#univariate AUC for all
y = df["label"].values
resultados = []

for col in numeric_cols:
    x = df[col].values
    try:
        auc = roc_auc_score(y, x)
        sep = max(auc, 1 - auc)
        resultados.append([col, auc, sep])
    except:
        pass

df_auc = pd.DataFrame(resultados, columns=["feature", "auc", "auc_separability"])
df_auc = df_auc.sort_values("auc_separability", ascending=False).reset_index(drop=True)

#filter auc
threshold_auc = 0.60
df_good = df_auc[df_auc["auc_separability"] > threshold_auc].copy().reset_index(drop=True)

print("\nFeatures with AUC separability > 0.60:", len(df_good))

features_boas = df_good["feature"].tolist()
auc_dict = dict(zip(df_good["feature"], df_good["auc_separability"]))

#correlation
threshold_corr = 0.90
corr = df[features_boas].corr(method="pearson", min_periods=30).abs()

#remove correlated
remover = set()

for i in range(len(features_boas)):
    for j in range(i + 1, len(features_boas)):

        f1 = features_boas[i]
        f2 = features_boas[j]

        if f1 in remover or f2 in remover:
            continue

        valor = corr.loc[f1, f2]

        if np.isfinite(valor) and valor >= threshold_corr:

            auc1 = auc_dict[f1]
            auc2 = auc_dict[f2]

            if auc1 >= auc2:
                remover.add(f2)
            else:
                remover.add(f1)

features = [f for f in features_boas if f not in remover]

df_final = df_good[df_good["feature"].isin(features)].copy()
df_final = df_final.sort_values("auc_separability", ascending=False).reset_index(drop=True)


print("Features correlated removed:", len(remover))
print("Final features:", len(features))

print("\nFinal features auc:")
print(df_final)


print(features)



In [ ]:
#plots
for col in features:

    x0 = df[df["label"] == 0][col].dropna()
    x1 = df[df["label"] == 1][col].dropna()



    # cut outliers
    x_all = df[col].dropna()
    lo = x_all.quantile(0.01)
    hi = x_all.quantile(0.99)

    x0 = x0[(x0 >= lo) & (x0 <= hi)]
    x1 = x1[(x1 >= lo) & (x1 <= hi)]



    #histogram
    plt.figure(figsize=(10, 4))
    plt.hist(x0, bins=60, range=(lo, hi), alpha=0.6, density=True, label="Label 0")
    plt.hist(x1, bins=60, range=(lo, hi), alpha=0.6, density=True, label="Label 1")
    plt.title(f"Histogram: {col}")
    plt.xlabel(col)
    plt.ylabel("Density")
    plt.legend()
    plt.tight_layout()
    plt.show()

    #boxplot
    plt.figure(figsize=(7, 4))
    plt.boxplot([x0.values, x1.values], labels=["0", "1"], showfliers=False)
    plt.title(f"Boxplot: {col}")
    plt.xlabel("Label")
    plt.ylabel(col)
    plt.tight_layout()
    plt.show()

In [ ]:
#heatmap
df_heat = df[features].dropna()


corr = df_heat.corr(method="pearson")

plt.figure(figsize=(14, 12))
plt.imshow(corr.values, vmin=-1, vmax=1, aspect="auto")
plt.xticks(range(len(features)), features, rotation=90)
plt.yticks(range(len(features)), features)
plt.title("Pearson Correlation (features AUC > 0.65)")
plt.colorbar()

for i in range(len(features)):
    for j in range(len(features)):
        val = corr.values[i, j]
        cor_texto = "white" if abs(val) > 0.5 else "black"
        plt.text(j, i, f"{val:.2f}", ha="center", va="center", color=cor_texto, fontsize=8)

plt.tight_layout()
plt.show()